# 05 · Pipeline và chia dữ liệu

Bài toán phân loại 11 lớp `Class`. Dùng dữ liệu raw để bộ tiền xử lý được fit lại bên trong từng fold. Chia theo nhóm `(Artist Name, Track Name)` và phân tầng theo nhãn bằng `StratifiedGroupKFold`; mỗi nhóm chỉ thuộc một fold. Fold đầu tiên tương ứng holdout validation xấp xỉ 20%.

Metric chính theo workflow nguồn: macro-F1; accuracy báo cáo kèm. Tập test chính thức không có nhãn nên không tham gia chọn mô hình.

In [1]:
from pathlib import Path
from datetime import datetime
import sys, csv, platform, time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score
from sklearn.base import clone

SEED = 42
RUN_START = datetime.now()
ROOT = Path(r"C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2")
DATA_DIR = ROOT / "DATA_CHALLENGE2"
LOG_DIR = ROOT / "logs"
NB_DIR = ROOT / "notebooks"
sys.path.insert(0, str(NB_DIR))
from preprocessing import MusicPreprocessor

TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test (2).csv"
TARGET = "Class"
ID = "Id"
GROUP = "group_id"
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy dữ liệu raw: {TRAIN_PATH}")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = train[TARGET].astype(int)
X = train.drop(columns=[TARGET])
# Cùng nghệ sĩ + tên bài hát luôn thuộc một fold; không dùng nhãn để tạo group.
group_key = (train["Artist Name"].fillna("").astype(str).str.strip().str.lower()
             + "||" + train["Track Name"].fillna("").astype(str).str.strip().str.lower())
groups = pd.factorize(group_key)[0]
CV = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
SPLITS = list(CV.split(X, y, groups=groups))

def make_pipeline(model):
    # MusicPreprocessor chỉ fit trên từng phần train của fold.
    return Pipeline([("preprocess", MusicPreprocessor()), ("model", model)])

def append_run_log(stage, notebook, goal, steps, notes):
    end = datetime.now()
    elapsed = round((end - RUN_START).total_seconds(), 1)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    run_id = f"{stage}_{end:%Y%m%d_%H%M%S}"
    slug = "Tô-Doanh-Quang-Khu"
    md_path = LOG_DIR / f"{stage}_{end:%Y%m%d_%H%M%S}_{slug}.md"
    suffix = 2
    while md_path.exists():
        md_path = LOG_DIR / f"{stage}_{end:%Y%m%d_%H%M%S}_{slug}_{suffix}.md"
        suffix += 1
    record = {"run_id":run_id,"date":end.strftime("%Y-%m-%d"),"time":end.strftime("%H:%M:%S"),
        "runner":"Tô Doanh Quang Khu","stage":stage,"notebook":notebook,"goal":goal,
        "steps":steps,"duration_sec":elapsed,"python":sys.version.split()[0],
        "platform":platform.platform(),"notes":notes + f"; markdown_log={md_path.name}"}
    fields=["run_id","date","time","runner","stage","notebook","goal","steps","duration_sec","python","platform","notes"]
    exists=(LOG_DIR/"experiment_log.csv").exists() and (LOG_DIR/"experiment_log.csv").stat().st_size>0
    with (LOG_DIR/"experiment_log.csv").open("a",newline="",encoding="utf-8-sig") as f:
        writer=csv.DictWriter(f,fieldnames=fields)
        if not exists: writer.writeheader()
        writer.writerow(record)
    steps_md="\n".join(f"- {part.strip()}" for part in steps.split(";"))
    md=f"""# Nhật ký thí nghiệm – {stage}

| Mục | Giá trị |
|---|---|
| Người chạy | Tô Doanh Quang Khu |
| Ngày chạy | {end.strftime('%d/%m/%Y')} |
| Giờ bắt đầu – kết thúc | {RUN_START.strftime('%H:%M:%S')} – {end.strftime('%H:%M:%S')} ({elapsed} giây) |
| Notebook | `{notebook}` |
| Mục tiêu | {goal} |
| Môi trường | Python {sys.version.split()[0]} · {platform.platform()} |

## Các bước & kết quả

{steps_md}

## Nhận xét

{notes}
"""
    md_path.write_text(md,encoding="utf-8")
    print(f"Đã ghi CSV và Markdown: {md_path}")
from sklearn.linear_model import LogisticRegression

print("Raw train/test:", train.shape, test.shape)
print("Số lớp:", y.nunique(), "| số track group:", pd.Series(groups).nunique())
print("Pipeline steps:", ["MusicPreprocessor (fit trong fold)", "LogisticRegression"])

fold_rows=[]
for fold,(tr,va) in enumerate(SPLITS,start=1):
    overlap=set(groups[tr]).intersection(set(groups[va]))
    assert not overlap, f"Group leakage ở fold {fold}"
    distribution=pd.concat([y.iloc[tr].value_counts(normalize=True).rename("train"),
                            y.iloc[va].value_counts(normalize=True).rename("validation")],axis=1).fillna(0)
    fold_rows.append({"fold":fold,"train_rows":len(tr),"valid_rows":len(va),
        "train_groups":pd.Series(groups[tr]).nunique(),"valid_groups":pd.Series(groups[va]).nunique(),
        "valid_classes":y.iloc[va].nunique(),"max_class_share_gap":(distribution["train"]-distribution["validation"]).abs().max()})
display(pd.DataFrame(fold_rows).round(4))

Raw train/test: (14396, 18) (3600, 17)
Số lớp: 11 | số track group: 13166
Pipeline steps: ['MusicPreprocessor (fit trong fold)', 'LogisticRegression']


,fold,train_rows,valid_rows,train_groups,valid_groups,valid_classes,max_class_share_gap
0,1,11516,2880,10533,2633,11,0.0003
1,2,11516,2880,10531,2635,11,0.0004
2,3,11517,2879,10533,2633,11,0.0003
3,4,11517,2879,10533,2633,11,0.0004
4,5,11518,2878,10534,2632,11,0.0002


## Holdout validation và pipeline

In [2]:
# Fold 1 được dùng làm holdout 80/20. StratifiedGroupKFold đồng thời giữ track group và gần đúng tỷ lệ lớp.
fit_idx, valid_idx = SPLITS[0]
X_fit, X_valid = X.iloc[fit_idx], X.iloc[valid_idx]
y_fit, y_valid = y.iloc[fit_idx], y.iloc[valid_idx]
model = make_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
model.fit(X_fit, y_fit)
pred = model.predict(X_valid)
holdout_metrics = {"macro_f1":f1_score(y_valid,pred,average="macro"),"accuracy":accuracy_score(y_valid,pred)}
print(f"Holdout: {len(fit_idx):,} fit / {len(valid_idx):,} validation rows")
print(f"Macro-F1={holdout_metrics['macro_f1']:.4f}; accuracy={holdout_metrics['accuracy']:.4f}")
assert set(groups[fit_idx]).isdisjoint(set(groups[valid_idx]))

Holdout: 11,516 fit / 2,880 validation rows
Macro-F1=0.5223; accuracy=0.4538


## Ghi nhật ký

In [3]:
split_text="; ".join(f"fold {r['fold']}: {r['train_rows']} train / {r['valid_rows']} validation rows, {r['valid_classes']} lớp, group-disjoint" for r in fold_rows)
append_run_log("05_Pipeline_Split","05_Pipeline_Split.ipynb","Stratified group split và pipeline tiền xử lý không rò rỉ",
    f"raw train={train.shape}; raw test={test.shape}; groups={pd.Series(groups).nunique()}; StratifiedGroupKFold 5 folds; holdout macro-F1={holdout_metrics['macro_f1']:.6f}; accuracy={holdout_metrics['accuracy']:.6f}; {split_text}",
    "MusicPreprocessor được fit riêng trong pipeline trên từng fold; test không dùng để chọn mô hình")

Đã ghi CSV và Markdown: C:\Users\Welcome\Desktop\ML cuối kỳ\MAY_HOC_NHOM\Challenge\Challenge 2\logs\05_Pipeline_Split_20261112_190957_Tô-Doanh-Quang-Khu.md
